In [ ]:
# Categories
# 1. Identifiers      => SUBJECT, SPELL_IDENTIFIER, ENCNTR_ID, ...
# 2. Dates / Raw text => ADMISSION_DATE, DISCHARGE_DATE, BIRTH_DATE, DEATH_DATE, ...
# 3. Target           =>
# 4. Features         =>

In [ ]:
# Missing
# IMD Decile -> Use an unknown category?
# Time since -> dslast, dsfirst,  ...
#  - Option 1: Fill missing with either -1 or 9999.
#  - Option 2: Pair with has prior event (hx) if using linear models.
#  - Tree models handle missing all right.

In [ ]:
# Encoding
# 1. Low cardinality (pd.get_dummies() or OneHotEcncoder.
#   MAIN_SPECIALTY_CODE_DESC
#   ETHNICITY_DES
#   GENDER_DESC

In [ ]:
# Splitting
# 1. Should we keep same patient in same fold? If evaluating temporal probably not?

In [ ]:
# %% [markdown]
# # 07 — Feature Preprocessing and Machine Learning Preparation

# %%
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from icare_risk.utils.io import load_pkg_yaml

schema = load_pkg_yaml("config/icare/schema.yaml")
DATA_DIR = Path(schema["data_root"])

# Load combined dataset from step 03
df = pd.read_parquet(DATA_DIR / "03_derived_analysis_dataset.parquet")
print(f"Loaded dataset shape: {df.shape}")

# %% [markdown]
# ## 1. Define Target and Feature Subsets
# %%
# Define target
TARGET = "esbl_binary" # Map esbl_status: 'ESBL' -> 1, 'non-ESBL'/'unknown' -> 0
df[TARGET] = (df["esbl_status"] == "ESBL").astype(int)

# Drop columns that are IDs, raw datetimes, or leak the label
ID_COLS = ["SUBJECT", "SPELL_IDENTIFIER", "ENCNTR_ID", "EPISODE_IDENTIFIER"]
DATETIME_COLS = [
    "ADMISSION_DATE", "DISCHARGE_DATE", "BIRTH_DATE", "DEATH_DATE",
    "UPDATE_DT_TM", "closest_culture_date", "index_admission", "ADMISSION_TIME"
]
TARGET_COLS = ["esbl_status", "esbl_label_test", TARGET]

# Feature columns
feature_cols = [c for c in df.columns if c not in ID_COLS + DATETIME_COLS + TARGET_COLS]

X = df[feature_cols].copy()
y = df[TARGET].copy()
groups = df["SUBJECT"].copy()

print(f"Total features selected: {X.shape[1]}")

# %% [markdown]
# ## 2. Handle Missing Values & Sentinel Imputations
# %%
# Example: Impute IMD Decile missing values
if "INDEX_OF_MULTIPLE_DEPRIVATION_DECILE" in X.columns:
    X["INDEX_OF_MULTIPLE_DEPRIVATION_DECILE"] = pd.to_numeric(
        X["INDEX_OF_MULTIPLE_DEPRIVATION_DECILE"], errors="coerce"
    )
    imd_median = X["INDEX_OF_MULTIPLE_DEPRIVATION_DECILE"].median()
    X["INDEX_OF_MULTIPLE_DEPRIVATION_DECILE"] = X["INDEX_OF_MULTIPLE_DEPRIVATION_DECILE"].fillna(imd_median)

# Handle clinical time-to-event / lookback NaNs (NaN = event never happened -> fill with large sentinel)
time_cols = [c for c in X.columns if "_dslast" in c or "_dsfirst" in c or "time_to_culture" in c or "days_since" in c]
X[time_cols] = X[time_cols].fillna(9999.0)

# Fill remaining numeric NaNs with 0 (e.g. counts)
numeric_cols = X.select_dtypes(include=[np.number]).columns
X[numeric_cols] = X[numeric_cols].fillna(0)

# Fill categorical NaNs with 'Missing'
cat_cols = X.select_dtypes(include=["object", "string"]).columns
X[cat_cols] = X[cat_cols].fillna("Missing")

# %% [markdown]
# ## 3. Categorical Encoding (One-Hot Encoding)
# %%
# Identify low-cardinality categorical features for OHE
categorical_features = [
    "MAIN_SPECIALTY_CODE_DESC",
    "ETHNICITY_DES",
    "GENDER_DESC",
    "INDEX_OF_MULTIPLE_DEPRIVATION_DECILE"
]
categorical_features = [c for c in categorical_features if c in X.columns]

numerical_features = [c for c in X.columns if c not in categorical_features]

print(f"Numerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Setup preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
    ]
)

# %% [markdown]
# ## 4. Patient-Aware Cross-Validation Split (StratifiedGroupKFold)
# %%
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups=groups)):
    print(f"Fold {fold}: Train size = {len(train_idx):,}, Val size = {len(val_idx):,}")
    # Verify zero patient overlap between train and val
    train_subjects = set(groups.iloc[train_idx])
    val_subjects = set(groups.iloc[val_idx])
    assert len(train_subjects.intersection(val_subjects)) == 0, f"Patient leakage detected in fold {fold}!"

print("Validation check passed: Zero patient overlap across all folds.")

# %% [markdown]
# ## 5. Save Cleaned Feature Matrix for Modeling
# %%
cleaned_output_path = DATA_DIR / "07_ml_ready_features.parquet"
# Combine X and y (keeping SUBJECT and SPELL_IDENTIFIER for tracking)
df_ml = X.copy()
df_ml["SUBJECT"] = groups.values
df_ml["SPELL_IDENTIFIER"] = df["SPELL_IDENTIFIER"].values
df_ml[TARGET] = y.values

df_ml.to_parquet(cleaned_output_path, index=False)
print(f"Saved ML-ready dataset to {cleaned_output_path} with shape {df_ml.shape}")